# 📊 JEE-LLM — Dataset Exploration

This notebook explores the JEE dataset: distribution of subjects, topics, difficulty levels, question types, and CoT quality metrics.

**Prerequisites**: Run `scripts/prepare_datasets.py` first.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
from collections import Counter

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('../dataset')
print('Available files:')
for f in sorted(DATA_DIR.glob('*.jsonl')):
    count = sum(1 for line in open(f) if line.strip())
    print(f'  {f.name:45s} {count:>8,}')

In [ ]:
# Load the main CoT dataset
cot_file = DATA_DIR / 'jee_cot_gpt4o.jsonl'
sample_file = DATA_DIR / 'samples' / 'sample_jee_cot.jsonl'

# Use sample if full dataset not available
target = cot_file if cot_file.exists() else sample_file
print(f'Loading from: {target}')

records = []
with open(target) as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f'Loaded {len(df):,} records')
df.head(3)

In [ ]:
# ── Subject Distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('JEE Dataset Distribution', fontsize=16, fontweight='bold', y=1.02)

# Subject
subj_counts = df['subject'].value_counts()
colors = ['#667eea', '#764ba2', '#f093fb']
axes[0].pie(subj_counts.values, labels=subj_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90,
            textprops={'fontsize': 11})
axes[0].set_title('By Subject', fontsize=13, fontweight='bold')

# Question Type
type_counts = df['question_type'].value_counts()
bars = axes[1].barh(type_counts.index, type_counts.values, color='#667eea', edgecolor='white')
axes[1].set_title('By Question Type', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Count')
for bar, v in zip(bars, type_counts.values):
    axes[1].text(v + 5, bar.get_y() + bar.get_height()/2,
                 f'{v:,}', va='center', fontsize=9)

# Difficulty
if 'difficulty' in df.columns:
    diff_counts = df['difficulty'].value_counts().sort_index()
    diff_colors = ['#2ecc71', '#27ae60', '#f39c12', '#e67e22', '#e74c3c']
    axes[2].bar(diff_counts.index.astype(str), diff_counts.values,
                color=diff_colors[:len(diff_counts)], edgecolor='white')
    axes[2].set_title('By Difficulty (1=Easy, 5=Hard)', fontsize=13, fontweight='bold')
    axes[2].set_xlabel('Difficulty Level')
    axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../docs/assets/dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Year-wise coverage ───────────────────────────────────────
if 'year' in df.columns:
    year_df = df.dropna(subset=['year'])
    year_subj = year_df.groupby(['year', 'subject']).size().unstack(fill_value=0)
    
    fig, ax = plt.subplots(figsize=(14, 5))
    year_subj.plot(kind='bar', ax=ax, color=colors[:len(year_subj.columns)],
                   edgecolor='white', width=0.7)
    ax.set_title('Problems by Year and Subject', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Number of Problems')
    ax.legend(title='Subject', loc='upper left')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
    
    print(f'Year range: {int(year_df.year.min())} – {int(year_df.year.max())}')
    print(f'Total years covered: {year_df.year.nunique()}')

In [ ]:
# ── CoT Quality Analysis ─────────────────────────────────────
df['q_len'] = df['question'].str.split().str.len()
df['cot_len'] = df['chain_of_thought'].fillna('').str.split().str.len()
df['has_boxed'] = df['chain_of_thought'].fillna('').str.contains(r'\\boxed{')
df['has_steps'] = df['chain_of_thought'].fillna('').str.contains(r'Step \d', regex=True)

print('── CoT Quality Metrics ──────────────────────────────')
print(f"Has CoT          : {(df['cot_len'] > 0).sum():,} ({(df['cot_len'] > 0).mean():.1%})")
print(f"Has \\boxed{{}}   : {df['has_boxed'].sum():,} ({df['has_boxed'].mean():.1%})")
print(f"Has step markers : {df['has_steps'].sum():,} ({df['has_steps'].mean():.1%})")
print(f"Avg Q length     : {df['q_len'].mean():.0f} words")
print(f"Avg CoT length   : {df['cot_len'].mean():.0f} words")
print(f"Min CoT length   : {df[df['cot_len']>0]['cot_len'].min()} words")
print(f"Max CoT length   : {df['cot_len'].max()} words")

# CoT length histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df[df['cot_len'] > 0]['cot_len'], bins=40, color='#667eea',
             edgecolor='white', alpha=0.85)
axes[0].set_title('CoT Length Distribution (words)', fontsize=13)
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['cot_len'].mean(), color='#e74c3c', linestyle='--',
                label=f"Mean: {df['cot_len'].mean():.0f}")
axes[0].legend()

cot_by_subj = df[df['cot_len'] > 0].groupby('subject')['cot_len'].mean()
axes[1].bar(cot_by_subj.index, cot_by_subj.values, color=colors, edgecolor='white')
axes[1].set_title('Avg CoT Length by Subject', fontsize=13)
axes[1].set_ylabel('Average Words')
for i, (subj, val) in enumerate(cot_by_subj.items()):
    axes[1].text(i, val + 5, f'{val:.0f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# ── Top Topics ───────────────────────────────────────────────
if 'topic' in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    subjects = ['Mathematics', 'Physics', 'Chemistry']

    for ax, subj, color in zip(axes, subjects, colors):
        subj_df = df[df['subject'] == subj]
        top_topics = subj_df['topic'].value_counts().head(10)
        bars = ax.barh(top_topics.index[::-1], top_topics.values[::-1],
                       color=color, alpha=0.85, edgecolor='white')
        ax.set_title(f'Top {subj} Topics', fontsize=12, fontweight='bold')
        ax.set_xlabel('Problem Count')
        for bar, v in zip(bars, top_topics.values[::-1]):
            ax.text(v + 0.5, bar.get_y() + bar.get_height()/2,
                    str(v), va='center', fontsize=8)

    plt.tight_layout()
    plt.savefig('../docs/assets/topic_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Sample Inspection ────────────────────────────────────────
print('── Sample Math Problem with CoT ────────────────────')
sample = df[(df['subject'] == 'Mathematics') & (df['cot_len'] > 100)].iloc[0]
print(f"ID      : {sample['id']}")
print(f"Year    : {sample.get('year', 'N/A')}")
print(f"Topic   : {sample.get('topic', 'N/A')}")
print(f"Type    : {sample.get('question_type', 'N/A')}")
print(f"\nQuestion:\n{sample['question'][:400]}")
print(f"\nChain of Thought (first 600 chars):\n{sample['chain_of_thought'][:600]}")
print(f"\nAnswer  : {sample['correct_answer']}")